In [ ]:
# imports
import sys
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

sys.path.append('helpers/pcca_fa/')
from dual_pfc_funcs import load_dict, getParams
import helpers.pcca_fa.pcca_fa_mdl as pf
import helpers.pcca_fa.cca.prob_cca as pcca

color_map = getParams()['color_map']
data_path = 'preprocessed_data/'
max_dim = 15

In [ ]:
# panel a: vary %sv
filename = data_path + 'simdataset_varySv_noWS_n1000.pkl'
dat = load_dict(filename)
sv_configs = dat['sv_list']
n_boots = dat['n_boots']

df1000 = pd.DataFrame(columns=['SimId','SvConfigAcross','SvConfigWithin','GT-SvW1','GT-SvW2','Est-SvW1','Est-SvW2','Error-SvW1','Error-SvW2','GT-SvL1','GT-SvL2','Est-SvL1','Est-SvL2','Error-SvL1','Error-SvL2'])
for i,config in enumerate(sv_configs):
    # get ground truth %sv
    gt_param = dat['sim_params'][i]
    mdl = pf.pcca_fa()
    mdl.set_params(gt_param)
    psv_gt = mdl.compute_psv()

    for boot in range(n_boots):
        idx = boot + (i*n_boots)

        # find estimated %sv
        est_param = dat['est_params'][idx]
        mdl = pf.pcca_fa()
        mdl.set_params(est_param)
        psv_est = mdl.compute_psv()

        df2 = {'SimId':idx,'SvConfigAcross':config[0],'SvConfigWithin':config[1],'GT-SvW1':psv_gt['avg_psv_W_1'],'GT-SvW2':psv_gt['avg_psv_W_2'],'Est-SvW1':psv_est['avg_psv_W_1'],'Est-SvW2':psv_est['avg_psv_W_2'],
                'Error-SvW1':psv_est['avg_psv_W_1']-psv_gt['avg_psv_W_1'],'Error-SvW2':psv_est['avg_psv_W_2']-psv_gt['avg_psv_W_2'],
                'GT-SvL1':psv_gt['avg_psv_L_1'],'GT-SvL2':psv_gt['avg_psv_L_2'],'Est-SvL1':psv_est['avg_psv_L_1'],'Est-SvL2':psv_est['avg_psv_L_2'],
                'Error-SvL1':psv_est['avg_psv_L_1']-psv_gt['avg_psv_L_1'],'Error-SvL2':psv_est['avg_psv_L_2']-psv_gt['avg_psv_L_2']}
        df1000.loc[len(df1000)] = df2

filename = data_path + 'simdataset_varySv_noWS_n300.pkl'
dat = load_dict(filename)
sv_configs = dat['sv_list']
n_boots = dat['n_boots']

df300 = pd.DataFrame(columns=['SimId','SvConfigAcross','SvConfigWithin','GT-SvW1','GT-SvW2','Est-SvW1','Est-SvW2','Error-SvW1','Error-SvW2','GT-SvL1','GT-SvL2','Est-SvL1','Est-SvL2','Error-SvL1','Error-SvL2'])
for i,config in enumerate(sv_configs):
    # get ground truth %sv
    gt_param = dat['sim_params'][i]
    mdl = pf.pcca_fa()
    mdl.set_params(gt_param)
    psv_gt = mdl.compute_psv()

    for boot in range(n_boots):
        idx = boot + (i*n_boots)

        # find estimated %sv
        est_param = dat['est_params'][idx]
        mdl = pf.pcca_fa()
        mdl.set_params(est_param)
        psv_est = mdl.compute_psv()

        df2 = {'SimId':idx,'SvConfigAcross':config[0],'SvConfigWithin':config[1],'GT-SvW1':psv_gt['avg_psv_W_1'],'GT-SvW2':psv_gt['avg_psv_W_2'],'Est-SvW1':psv_est['avg_psv_W_1'],'Est-SvW2':psv_est['avg_psv_W_2'],
                'Error-SvW1':psv_est['avg_psv_W_1']-psv_gt['avg_psv_W_1'],'Error-SvW2':psv_est['avg_psv_W_2']-psv_gt['avg_psv_W_2'],
                'GT-SvL1':psv_gt['avg_psv_L_1'],'GT-SvL2':psv_gt['avg_psv_L_2'],'Est-SvL1':psv_est['avg_psv_L_1'],'Est-SvL2':psv_est['avg_psv_L_2'],
                'Error-SvL1':psv_est['avg_psv_L_1']-psv_gt['avg_psv_L_1'],'Error-SvL2':psv_est['avg_psv_L_2']-psv_gt['avg_psv_L_2']}
        df300.loc[len(df300)] = df2

df300

In [ ]:
plot_area = '1' # which area to plot (1 or 2)

fig,ax = plt.subplots(2,1,sharex=True,sharey=True,figsize=(4,6))
fig.tight_layout(pad=2.5)

fig.supylabel('estimated across-area %sv', color=color_map['across'])
fig.supxlabel('estimated within-area %sv', color=color_map['within'])

ax[0].plot([0,np.max(sv_configs)],[0,np.max(sv_configs)],'--', color='gray')
ax[1].plot([0,np.max(sv_configs)],[0,np.max(sv_configs)],'--', color='gray') 

xdata = df1000.groupby(['SvConfigAcross'])['GT-SvL{}'.format(plot_area)].mean().to_list()
ydata = df1000.groupby(['SvConfigAcross'])['GT-SvW{}'.format(plot_area)].mean().to_list()
ax[1].scatter(xdata,ydata,s=100,color='gray', marker='o')

xdata = df1000.groupby(['SvConfigAcross'])['Est-SvL{}'.format(plot_area)].mean().to_list()
ydata = df1000.groupby(['SvConfigAcross'])['Est-SvW{}'.format(plot_area)].mean().to_list()
xerr = df1000.groupby(['SvConfigAcross'])['Est-SvL{}'.format(plot_area)].std().to_list()
yerr = df1000.groupby(['SvConfigAcross'])['Est-SvW{}'.format(plot_area)].std().to_list()
ax[1].errorbar(xdata, ydata, yerr=yerr, xerr=xerr, fmt='o', color='k', ms=4, label='pCCA-FA')

xdata = df300.groupby(['SvConfigAcross'])['GT-SvL{}'.format(plot_area)].mean().to_list()
ydata = df300.groupby(['SvConfigAcross'])['GT-SvW{}'.format(plot_area)].mean().to_list()
ax[0].scatter(xdata,ydata,s=100,color='gray', marker='o')
xdata = df300.groupby(['SvConfigAcross'])['Est-SvL{}'.format(plot_area)].mean().to_list()
ydata = df300.groupby(['SvConfigAcross'])['Est-SvW{}'.format(plot_area)].mean().to_list()
xerr = df300.groupby(['SvConfigAcross'])['Est-SvL{}'.format(plot_area)].std().to_list()
yerr = df300.groupby(['SvConfigAcross'])['Est-SvW{}'.format(plot_area)].std().to_list()
ax[0].errorbar(xdata, ydata, yerr=yerr, xerr=xerr, fmt='o', color='k', ms=4, label='pCCA-FA')

ax[0].set_title('N=300')
ax[1].set_title('N=1000')
ax[0].set_aspect('equal')
ax[1].set_aspect('equal')

ax[0].set_yticks(np.arange(0,21,5))
ax[0].set_xticks(np.arange(0,21,5))

if SAVE_FIG:
    pdf = PdfPages('figs/sv_error_varySv_noWS.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

# prove other area is similar in quality
other_area = '2' if plot_area == '1' else '1'
df1000.groupby(['SvConfigAcross']).agg({'GT-SvW{}'.format(other_area):  ['mean'],
                                        'Est-SvW{}'.format(other_area): ['mean', np.std],
                                        'GT-SvL{}'.format(other_area):  ['mean'],
                                        'Est-SvL{}'.format(other_area): ['mean', np.std]})

In [ ]:
# panel b: vary dim
filename = data_path + 'simdataset_varyDim_noWS_n1000.pkl'
dat = load_dict(filename)
dim_configs = dat['dim_list']
n_boots = dat['n_boots']

df1000 = pd.DataFrame(columns=['SimId','DimConfigAcross','DimConfigWithin','GT-DSharedW1','GT-DSharedW2','Est-DSharedW1','Est-DSharedW2','Error-DSharedW1','Error-DSharedW2','GT-DSharedL1','GT-DSharedL2','Est-DSharedL1','Est-DSharedL2','Error-DSharedL1','Error-DSharedL2'])
for i,config in enumerate(dim_configs):
    # get ground truth %sv
    gt_param = dat['sim_params'][i]
    mdl = pf.pcca_fa()
    mdl.set_params(gt_param)
    dim_gt = mdl.compute_dshared()

    for boot in range(n_boots):
        idx = boot + (i*n_boots)

        # find estimated %sv
        est_param = dat['est_params'][idx]
        mdl = pf.pcca_fa()
        mdl.set_params(est_param)
        dim_est = mdl.compute_dshared()

        df2 = {'SimId':idx,'DimConfigAcross':config[0],'DimConfigWithin':config[1],'GT-DSharedW1':dim_gt['dshared_W_1'],'GT-DSharedW2':dim_gt['dshared_W_2'],
            'Est-DSharedW1':dim_est['dshared_W_1'],'Est-DSharedW2':dim_est['dshared_W_2'],'Error-DSharedW1':dim_est['dshared_W_1']-dim_gt['dshared_W_1'],'Error-DSharedW2':dim_est['dshared_W_2']-dim_gt['dshared_W_2'],
            'GT-DSharedL1':dim_gt['dshared_L_1'],'GT-DSharedL2':dim_gt['dshared_L_2'],'Est-DSharedL1':dim_est['dshared_L_1'],'Est-DSharedL2':dim_est['dshared_L_2'],
            'Error-DSharedL1':dim_est['dshared_L_1']-dim_gt['dshared_L_1'],'Error-DSharedL2':dim_est['dshared_L_2']-dim_gt['dshared_L_2']}
        df1000.loc[len(df1000)] = df2

filename = data_path + 'simdataset_varyDim_noWS_n300.pkl'
dat = load_dict(filename)
dim_configs = dat['dim_list']
n_boots = dat['n_boots']

df300 = pd.DataFrame(columns=['SimId','DimConfigAcross','DimConfigWithin','GT-DSharedW1','GT-DSharedW2','Est-DSharedW1','Est-DSharedW2','Error-DSharedW1','Error-DSharedW2','GT-DSharedL1','GT-DSharedL2','Est-DSharedL1','Est-DSharedL2','Error-DSharedL1','Error-DSharedL2'])
for i,config in enumerate(dim_configs):
    # get ground truth %sv
    gt_param = dat['sim_params'][i]
    mdl = pf.pcca_fa()
    mdl.set_params(gt_param)
    dim_gt = mdl.compute_dshared()

    for boot in range(n_boots):
        idx = boot + (i*n_boots)

        # find estimated %sv
        est_param = dat['est_params'][idx]
        mdl = pf.pcca_fa()
        mdl.set_params(est_param)
        dim_est = mdl.compute_dshared()

        df2 = {'SimId':idx,'DimConfigAcross':config[0],'DimConfigWithin':config[1],'GT-DSharedW1':dim_gt['dshared_W_1'],'GT-DSharedW2':dim_gt['dshared_W_2'],
            'Est-DSharedW1':dim_est['dshared_W_1'],'Est-DSharedW2':dim_est['dshared_W_2'],'Error-DSharedW1':dim_est['dshared_W_1']-dim_gt['dshared_W_1'],'Error-DSharedW2':dim_est['dshared_W_2']-dim_gt['dshared_W_2'],
            'GT-DSharedL1':dim_gt['dshared_L_1'],'GT-DSharedL2':dim_gt['dshared_L_2'],'Est-DSharedL1':dim_est['dshared_L_1'],'Est-DSharedL2':dim_est['dshared_L_2'],
            'Error-DSharedL1':dim_est['dshared_L_1']-dim_gt['dshared_L_1'],'Error-DSharedL2':dim_est['dshared_L_2']-dim_gt['dshared_L_2']}
        df300.loc[len(df300)] = df2

df300

In [ ]:
plot_area = '2'

fig,ax = plt.subplots(2,1,sharex=True,sharey=True,figsize=(4,6))
fig.tight_layout(pad=2.5)

fig.supylabel('estimated across-area d_shared', color=color_map['across'])
fig.supxlabel('estimated within-area d_shared', color=color_map['within'])

ax[0].plot([0,np.max(dim_configs)],[0,np.max(dim_configs)],'--', color='gray')
ax[1].plot([0,np.max(dim_configs)],[0,np.max(dim_configs)],'--', color='gray') 

xdata = df1000.groupby(['DimConfigAcross'])['GT-DSharedL{}'.format(plot_area)].mean().to_list()
ydata = df1000.groupby(['DimConfigAcross'])['GT-DSharedW{}'.format(plot_area)].mean().to_list()
ax[1].scatter(xdata,ydata,s=100,color='gray', marker='o')

xdata = df1000.groupby(['DimConfigAcross'])['Est-DSharedL{}'.format(plot_area)].mean().to_list()
ydata = df1000.groupby(['DimConfigAcross'])['Est-DSharedW{}'.format(plot_area)].mean().to_list()
xerr = df1000.groupby(['DimConfigAcross'])['Est-DSharedL{}'.format(plot_area)].std().to_list()
yerr = df1000.groupby(['DimConfigAcross'])['Est-DSharedW{}'.format(plot_area)].std().to_list()
ax[1].errorbar(xdata, ydata, yerr=yerr, xerr=xerr, fmt='o', color='k', ms=4, label='pCCA-FA')

xdata = df300.groupby(['DimConfigAcross'])['GT-DSharedL{}'.format(plot_area)].mean().to_list()
ydata = df300.groupby(['DimConfigAcross'])['GT-DSharedW{}'.format(plot_area)].mean().to_list()
ax[0].scatter(xdata,ydata,s=100,color='gray', marker='o')
xdata = df300.groupby(['DimConfigAcross'])['Est-DSharedL{}'.format(plot_area)].mean().to_list()
ydata = df300.groupby(['DimConfigAcross'])['Est-DSharedW{}'.format(plot_area)].mean().to_list()
xerr = df300.groupby(['DimConfigAcross'])['Est-DSharedL{}'.format(plot_area)].std().to_list()
yerr = df300.groupby(['DimConfigAcross'])['Est-DSharedW{}'.format(plot_area)].std().to_list()
ax[0].errorbar(xdata, ydata, yerr=yerr, xerr=xerr, fmt='o', color='k', ms=4, label='pCCA-FA')

ax[0].set_title('N=300')
ax[1].set_title('N=1000')
ax[0].set_aspect('equal')
ax[1].set_aspect('equal')

ax[0].set_yticks(np.arange(np.max(dim_configs)+1))
ax[0].set_xticks(np.arange(np.max(dim_configs)+1))

if SAVE_FIG:
    pdf = PdfPages('figs/dim_error_varyDim_noWS.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

# prove other area is similar in quality
other_area = '2' if plot_area == '1' else '1'
df1000.groupby(['DimConfigAcross']).agg({'GT-DSharedW{}'.format(other_area):  ['mean'],
                                         'Est-DSharedW{}'.format(other_area): ['mean', np.std],
                                         'GT-DSharedL{}'.format(other_area):  ['mean'],
                                         'Est-DSharedL{}'.format(other_area): ['mean', np.std]})

In [ ]:
# panel e: vary N
filename = data_path + 'simdataset_varyN_noWS.pkl'
dat = load_dict(filename)
n_boots = dat['n_boots']
n_trials = dat['N']

df = pd.DataFrame(columns=['SimId','NConfig','GT-SvW1','GT-SvW2','Est-SvW1','Est-SvW2','Error-SvW1','Error-SvW2',
                           'GT-SvL1','GT-SvL2','Est-SvL1','Est-SvL2','Error-SvL1','Error-SvL2',
                           'pCCA-SvW1', 'pCCA-SvW2','pCCA-Error-SvW1','pCCA-Error-SvW2',
                           'GT-DSharedW1','GT-DSharedW2','Est-DSharedW1','Est-DSharedW2','Error-DSharedW1','Error-DSharedW2',
                           'GT-DSharedL1','GT-DSharedL2','Est-DSharedL1','Est-DSharedL2','Error-DSharedL1','Error-DSharedL2',
                           'pCCA-DSharedW1', 'pCCA-DSharedW2','pCCA-Error-DSharedW1','pCCA-Error-DSharedW2'])
for idx in range(len(dat['sim_params'])):
    gt_param = dat['sim_params'][idx]
    est_param = dat['est_params'][idx]
    pcca_param = dat['pcca_params'][idx]
    
    # get ground truth %sv
    mdl = pf.pcca_fa()
    mdl.set_params(gt_param)
    psv_gt = mdl.compute_psv()
    dim_gt = mdl.compute_dshared()

    # get est %sv for pCCA-FA
    mdl = pf.pcca_fa()
    mdl.set_params(est_param)
    psv_est = mdl.compute_psv()
    dim_est = mdl.compute_dshared()

    # get est %sv for pCCA :o
    mdl = pcca.prob_cca()
    mdl.set_params(pcca_param)
    psv_pcca = mdl.compute_psv()
    dim_pcca = mdl.compute_dshared()

    df2 = {'SimId':idx,'NConfig':n_trials[idx],
           'GT-SvW1':psv_gt['avg_psv_W_1'],'GT-SvW2':psv_gt['avg_psv_W_2'],'Est-SvW1':psv_est['avg_psv_W_1'],'Est-SvW2':psv_est['avg_psv_W_2'],
           'GT-SvL1':psv_gt['avg_psv_L_1'],'GT-SvL2':psv_gt['avg_psv_L_2'],'Est-SvL1':psv_est['avg_psv_L_1'],'Est-SvL2':psv_est['avg_psv_L_2'],
            'Error-SvW1':psv_est['avg_psv_W_1']-psv_gt['avg_psv_W_1'],'Error-SvW2':psv_est['avg_psv_W_2']-psv_gt['avg_psv_W_2'],
            'Error-SvL1':psv_est['avg_psv_L_1']-psv_gt['avg_psv_L_1'],'Error-SvL2':psv_est['avg_psv_L_2']-psv_gt['avg_psv_L_2'],
            'pCCA-SvW1':psv_pcca['psv_x'], 'pCCA-SvW2':psv_pcca['psv_y'], 'pCCA-Error-SvW1':psv_pcca['psv_x']-psv_gt['avg_psv_W_1'], 'pCCA-Error-SvW2':psv_pcca['psv_y']-psv_gt['avg_psv_W_2'],
            'GT-DSharedW1':dim_gt['dshared_W_1'],'GT-DSharedW2':dim_gt['dshared_W_2'],'Est-DSharedW1':dim_est['dshared_W_1'],'Est-DSharedW2':dim_est['dshared_W_2'],
            'GT-DSharedL1':dim_gt['dshared_L_1'],'GT-DSharedL2':dim_gt['dshared_L_2'],'Est-DSharedL1':dim_est['dshared_L_1'],'Est-DSharedL2':dim_est['dshared_L_2'],
            'Error-DSharedW1':dim_est['dshared_W_1']-dim_gt['dshared_W_1'],'Error-DSharedW2':dim_est['dshared_W_2']-dim_gt['dshared_W_2'],
            'Error-DSharedL1':dim_est['dshared_L_1']-dim_gt['dshared_L_1'],'Error-DSharedL2':dim_est['dshared_L_2']-dim_gt['dshared_L_2'],
            'pCCA-DSharedW1':dim_pcca['dshared_x'], 'pCCA-DSharedW2':dim_pcca['dshared_y'], 'pCCA-Error-DSharedW1':dim_pcca['dshared_x']-dim_gt['dshared_W_1'], 'pCCA-Error-DSharedW2':dim_pcca['dshared_y']-dim_gt['dshared_W_2']}
    df.loc[len(df)] = df2

sv_global_mean, sv_local_mean, pcca_mean = [], [], []
sv_global_std, sv_local_std, pcca_std = [], [], []
dim_global_mean, dim_local_mean, dim_pcca_mean = [], [], []
dim_global_std, dim_local_std, dim_pcca_std = [], [], []
for n in np.unique(n_trials):
    filt = df['NConfig'] == n

    # calculate error in estimated shared variance
    curr_global = np.concatenate((df[filt]['Error-SvW1'],df[filt]['Error-SvW2']))
    curr_local = np.concatenate((df[filt]['Error-SvL1'],df[filt]['Error-SvL2']))
    curr_pcca = np.concatenate((df[filt]['pCCA-Error-SvW1'],df[filt]['pCCA-Error-SvW2']))

    sv_global_mean.append(np.mean(curr_global))
    sv_local_mean.append(np.mean(curr_local))
    pcca_mean.append(np.mean(curr_pcca))
    sv_global_std.append(np.std(curr_global))
    sv_local_std.append(np.std(curr_local))
    pcca_std.append(np.std(curr_pcca))

    # calculate error in estimated dimensionality
    curr_global = np.concatenate((df[filt]['Error-DSharedW1'],df[filt]['Error-DSharedW2']))
    curr_local = np.concatenate((df[filt]['Error-DSharedL1'],df[filt]['Error-DSharedL2']))
    curr_pcca = np.concatenate((df[filt]['pCCA-Error-DSharedW1'],df[filt]['pCCA-Error-DSharedW2']))

    dim_global_mean.append(np.mean(curr_global))
    dim_local_mean.append(np.mean(curr_local))
    dim_pcca_mean.append(np.mean(curr_pcca))
    dim_global_std.append(np.std(curr_global))
    dim_local_std.append(np.std(curr_local))
    dim_pcca_std.append(np.std(curr_pcca))

In [ ]:
fig,ax = plt.subplots(2,1,sharex=True,sharey=True,figsize=(5,6))
fig.tight_layout(pad=2.5)

fig.supylabel('error in %sv')

ax[0].plot([0,np.max(n_trials)],[0,0],'--', color='gray',label='ground truth') # across
ax[1].plot([0,np.max(n_trials)],[0,0],'--', color='gray') # within

xdata = np.unique(n_trials)
ax[0].errorbar(xdata, sv_global_mean, yerr=sv_global_std, fmt='-o', color=color_map['across'], ms=4, label='pCCA-FA')
ax[0].errorbar(xdata, pcca_mean, yerr=pcca_std, fmt='-o', color=color_map['across'], alpha=0.5, ms=4,label='pCCA')
ax[1].errorbar(xdata, sv_local_mean, yerr=sv_local_std, fmt='-o', color=color_map['within'], ms=4)

ax[0].set_xticks(np.arange(0,np.max(n_trials)+1,500))
ax[0].legend()

if SAVE_FIG:
    pdf = PdfPages('figs/sv_error_varyN_noWS.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()
    

fig,ax = plt.subplots(2,1,sharex=True,sharey=True,figsize=(5,6))
fig.tight_layout(pad=2.5)

fig.supylabel('error in d_shared')

ax[0].plot([0,np.max(n_trials)],[0,0],'--', color='gray',label='ground truth') # across
ax[1].plot([0,np.max(n_trials)],[0,0],'--', color='gray') # within

xdata = np.unique(n_trials)
ax[0].errorbar(xdata, dim_global_mean, yerr=dim_global_std, fmt='-o', color=color_map['across'], ms=4, label='pCCA-FA')
ax[0].errorbar(xdata, dim_pcca_mean, yerr=dim_pcca_std, fmt='-o', color=color_map['across'], alpha=0.5, ms=4,label='pCCA')
ax[1].errorbar(xdata, dim_local_mean, yerr=dim_local_std, fmt='-o', color=color_map['within'], ms=4)

ax[0].set_xticks(np.arange(0,np.max(n_trials)+1,500))
ax[0].legend()

if SAVE_FIG:
    pdf = PdfPages('figs/dim_error_varyN_noWS.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()